<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.6: 生成器: 类型
**上一节: [面向对象编程](3.5_object_oriented_programming.ipynb)**<br>
**下一节: [FIRRTL 介绍](4.1_firrtl_ast.ipynb)**

## 动机
Scala 是一种强类型编程语言。
这是一把双刃剑；一方面，许多在 Python（一种动态类型语言）中能够编译和执行的程序在 Scala 中会在编译时失败。
另一方面，在 Scala 中编译的程序比类似的 Python 程序包含的运行时错误要少得多。

在本节中，我们的目标是让你熟悉类型作为 Scala 中的一等公民。
虽然最初你可能会觉得生产力有限，但你很快就能学会理解编译时错误消息，以及如何设计你的程序，利用类型系统为你捕获更多错误。


## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 静态类型<a name="types-in-scala"></a>

## Scala 中的类型

Scala 中的所有对象都有一个类型，通常是该对象的类。
让我们看一些示例：

In [ ]:
println(10.getClass)
println(10.0.getClass)
println("ten".getClass)

当您声明自己的类时，它具有关联的类型。

In [ ]:
class MyClass {
    def myMethod = ???
}
println(new MyClass().getClass)

虽然不是必需的，但强烈建议您**为所有函数声明定义输入和输出类型**。
这将让 Scala 编译器捕获函数的错误使用。

In [ ]:
def double(s: String): String = s + s
// 取消注释以下代码来测试它
// double("hi")      // double 的正确使用
// double(10)        // 错误的输入参数！
// double("hi") / 10 // double 输出的错误使用！

不返回任何内容的函数返回类型 `Unit`。

In [ ]:
var counter = 0
def increment(): Unit = {
    counter += 1
}
increment()

## Scala vs. Chisel Types<a name="scala-vs-Chisel-types"></a>

回顾：模块 2.2 讨论了 Chisel 类型和 Scala 类型之间的区别，例如，以下事实：
```scala
val a = 导线(UInt(4.W))
a := 0.U
```
是合法的，因为 `0.U` 是类型 `UInt`（一个 Chisel 类型），而
```scala
val a = 导线(UInt(4.W))
a := 0
```
是非法的，因为 0 是类型 `Int`（一个 Scala 类型）。

这也适用于 `Bool`，这是一个与 `Boolean` 不同的 Chisel 类型。
```scala
val bool = 导线(Bool())
val boolean: Boolean = false
// legal
when (bool) { ... }
if (boolean) { ... }
// illegal
if (bool) { ... }
when (boolean) { ... }
```

如果您犯了一个错误，混合使用 `UInt` 和 `Int` 或 `Bool` 和 `Boolean`，Scala 编译器通常会为您捕获这些错误。
这是因为 Scala 的静态类型。
在编译时，编译器能够区分 Chisel 和 Scala 类型，并且能够理解 `if ()` 期望一个 `Boolean` 而 `when ()` 期望一个 `Bool`。


## Scala 类型 Coercion<a name="类型-coercion"></a>

<!-- typeOf. Scala 有一个名为 `typeOf[T]` 的函数，它返回 `T` 的类型对象。 -->
<!-- 这对 Chisel 用户来说实际上似乎没有用... -->

### asInstanceOf

`x.asInstanceOf[T]` 将对象 `x` 转换为类型 `T`。如果给定对象无法转换为类型 `T`，则会抛出异常。

In [ ]:
val x: UInt = 3.U
try {
  println(x.asInstanceOf[Int])
} catch {
  case e: java.lang.ClassCastException => println("As expected, we can't cast UInt to Int")
}

// 但我们可以将 UInt 转换为 Data，因为 UInt 继承自 Data。
println(x.asInstanceOf[Data])


### 类型 Casting in Chisel

如果您尝试运行下面的代码而不删除注释，将会出现错误。
问题是什么？
它试图将 `UInt` 赋值给 `SInt`，这是非法的。

Chisel 有一组类型转换函数。
最通用的是 `asTypeOf()`，如下所示。
一些 Chisel 对象还定义了 `asUInt()` 和 `asSInt()` 以及其他一些函数。

如果从下面的代码块中移除 `//`，该示例应该对您有效。


In [ ]:
class TypeConvertDemo extends Module {
    val io = IO(new Bundle {
        val in  = Input(UInt(4.W))
        val out = Output(SInt(4.W))
    })
    io.out := io.in//.asTypeOf(io.out)
}

test(new TypeConvertDemo) { c =>
      c.io.in.poke(3.U)
      c.io.out.expect(3.S)
      c.io.in.poke(15.U)
      c.io.out.expect(-1.S)
}

---
# 类型 Matching<a name="类型-matching"></a>

## 匹配运算符
回想一下，在 3.1 中介绍了匹配运算符。
类型匹配在尝试编写类型通用的生成器时特别有用。
以下示例展示了一个可以添加两个类型为 `UInt` 或 `SInt` 的字面量的"生成器"。
后续章节将更多地讨论编写类型通用的生成器。

**注意：在 Scala 中有更好、更安全的方式来编写类型通用的生成器**。

In [ ]:
class ConstantSum(in1: Data, in2: Data) extends Module {
    val io = IO(new Bundle {
        val out = Output(chiselTypeOf(in1)) // in case in1 is literal then just get its type
    })
    (in1, in2) match {
        case (x: UInt, y: UInt) => io.out := x + y
        case (x: SInt, y: SInt) => io.out := x + y
        case _ => throw new Exception("I give up!")
    }
}
println(getVerilog(dut = new ConstantSum(3.U, 4.U)))
println(getVerilog(dut = new ConstantSum(-3.S, 4.S)))
println(getVerilog(dut = new ConstantSum(3.U, 4.S)))


需要记住的是，Chisel 类型通常不应该进行值匹配。
Scala 的匹配在电路详细化期间执行，但您可能想要的是详细化后的比较。
以下给出了语法错误：

In [ ]:
class InputIsZero extends Module {
    val io = IO(new Bundle {
        val in  = Input(UInt(16.W))
        val out = Output(Bool())
    })
    io.out := (io.in match {
        // note that case 0.U is an error
        case (0.U) => true.B
        case _   => false.B
    })
}
println(getVerilog(new InputIsZero))

## Unapply
当你进行匹配时，实际上发生了什么？
Scala 如何让你像这样使用 case 类进行花哨的值匹配：
```scala
case 类 Something(a: String, b: Int)
val a = Something("A", 3)
a match {
    case Something("A", 值) => 值
    case Something(str, 3)     => 0
}
```

事实证明，为每个 case 类创建的伴生对象除了包含 **apply** 方法外，还包含一个 **unapply** 方法。
什么是 **unapply** 方法？

Scala unapply 方法是另一种语法糖，它赋予匹配语句在匹配时既匹配类型又**提取值**的能力。

让我们看看以下示例。
出于某种原因，假设如果生成器正在流水线化，延迟是 `3*totalWidth`，否则延迟是 `2*someOtherWidth`。
因为 case 类定义了 **unapply**，我们可以匹配 case 类内部的值，如下所示：

In [ ]:
case class SomeGeneratorParameters(
    someWidth: Int,
    someOtherWidth: Int = 10,
    pipelineMe: Boolean = false
) {
    require(someWidth >= 0)
    require(someOtherWidth >= 0)
    val totalWidth = someWidth + someOtherWidth
}

def delay(p: SomeGeneratorParameters): Int = p match {
    case SomeGeneratorParameters(_, sw, false) => sw * 2
    case sg @SomeGeneratorParameters(_, _, true) => sg.totalWidth * 3
}

println(delay(SomeGeneratorParameters(10, 10)))
println(delay(SomeGeneratorParameters(10, 10, true)))

如果你查看 `delay` 函数，你应该注意到除了匹配每个字符的类型之外，我们还要：
- 直接引用参数的内部值
- 有时，直接匹配参数的内部值

这些之所以可能，是因为编译器实现了一个 `unapply` 方法。请注意，解构 case 只是语法糖；例如，以下两个案例是等价的：
```scala
case p: SomeGeneratorParameters => p.sw * 2
case SomeGeneratorParameters(_, sw, _) => sw * 2
```

此外，还有更多的匹配语法和风格。以下两个案例也是等价的，但第二个允许您匹配内部值，同时仍然引用父值：
```scala
case SomeGeneratorParameters(_, sw, true) => sw
case sg @SomeGeneratorParameters(_, sw, true) => sw
```

最后，您可以直接将条件检查嵌入到匹配语句中，如以下第三个等效示例所示：
```scala
case SomeGeneratorParameters(_, sw, false) => sw * 2
case s @SomeGeneratorParameters(_, sw, false) => s.sw * 2
case s: SomeGeneratorParameters if s.pipelineMe => s.sw * 2
```

所有这些语法都是由 Scala unapply 方法启用的，该方法包含在类的伴生对象中。如果您想要解构一个类但不想将其设为 case 类，可以手动实现 unapply 方法。以下示例演示了如何手动实现类的 apply 和 unapply 方法：

In [ ]:
class Boat(val name: String, val length: Int)
object Boat {
    def unapply(b: Boat): Option[(String, Int)] = Some((b.name, b.length))
    def apply(name: String, length: Int): Boat = new Boat(name, length)
}

def getSmallBoats(seq: Seq[Boat]): Seq[Boat] = seq.filter { b =>
    b match {
        case Boat(_, length) if length < 60 => true
        case Boat(_, _) => false
    }
}

val boats = Seq(Boat("Santa Maria", 62), Boat("Pinta", 56), Boat("Nina", 50))
println(getSmallBoats(boats).map(_.name).mkString(" and ") + " are small boats!")

## 部分函数
这是一个简要概述；[本指南](https://twitter.github.io/scala_school/pattern-matching-and-functional-composition.html#PartialFunction)有更详细的概述。

部分函数是仅在其输入的子集上定义的函数。
就像一个选项，部分函数可能对特定输入没有值。
这可以通过 `isDefinedAt(...)` 进行测试。

部分函数可以通过 `orElse` 链接在一起。

请注意，使用未定义的输入调用 `PartialFunction` 将导致运行时错误。例如，如果 `PartialFunction` 的输入是用户定义的，则可能发生这种情况。为了更类型安全，我们建议编写返回 `Option` 的函数。

In [ ]:
// 辅助函数，使此单元格不那么繁琐。
def printAndAssert(cmd: String, result: Boolean, expected: Boolean): Unit = {
  println(s"$cmd = $result")
  assert(result == expected)
}

// Defined for -1, 2, 5, etc.
val partialFunc1: PartialFunction[Int, String] = {
  case i if (i + 1) % 3 == 0 => "Something"
}
printAndAssert("partialFunc1.isDefinedAt(2)", partialFunc1.isDefinedAt(2), true)
printAndAssert("partialFunc1.isDefinedAt(5)", partialFunc1.isDefinedAt(5), true)
printAndAssert("partialFunc1.isDefinedAt(1)", partialFunc1.isDefinedAt(1), false)
printAndAssert("partialFunc1.isDefinedAt(0)", partialFunc1.isDefinedAt(0), false)
println(s"partialFunc1(2) = ${partialFunc1(2)}")
try {
  println(partialFunc1(0))
} catch {
  case e: scala.MatchError => println("partialFunc1(0) = can't apply PartialFunctions where they are not defined")
}

// Defined for 1, 4, 7, etc.
val partialFunc2: PartialFunction[Int, String] = {
  case i if (i + 2) % 3 == 0 => "Something else"
}
printAndAssert("partialFunc2.isDefinedAt(1)", partialFunc2.isDefinedAt(1), true)
printAndAssert("partialFunc2.isDefinedAt(0)", partialFunc2.isDefinedAt(0), false)
println(s"partialFunc2(1) = ${partialFunc2(1)}")
try {
  println(partialFunc2(0))
} catch {
  case e: scala.MatchError => println("partialFunc2(0) = can't apply PartialFunctions where they are not defined")
}

val partialFunc3 = partialFunc1 orElse partialFunc2
printAndAssert("partialFunc3.isDefinedAt(0)", partialFunc3.isDefinedAt(0), false)
printAndAssert("partialFunc3.isDefinedAt(1)", partialFunc3.isDefinedAt(1), true)
printAndAssert("partialFunc3.isDefinedAt(2)", partialFunc3.isDefinedAt(2), true)
printAndAssert("partialFunc3.isDefinedAt(3)", partialFunc3.isDefinedAt(3), false)
println(s"partialFunc3(1) = ${partialFunc3(1)}")
println(s"partialFunc3(2) = ${partialFunc3(2)}")

---
# 类型 Safe Connections<a name="类型-safe-connections"></a>

Chisel can check the 类型 for many connections, including:
* Bool/UInt to 时钟

对于其他类型，Chisel允许您连接它们，但可能会适当地截断/填充位。
* Bool/UInt 到 Bool/UInt
* 束 到 束

In [ ]:
class Bundle1 extends Bundle {
  val a = UInt(8.W)
}

class Bundle2 extends Bundle1 {
  val b = UInt(16.W)
}

class BadTypeModule extends Module {
  val io = IO(new Bundle {
    val c  = Input(Clock())
    val in = Input(UInt(2.W))
    val out = Output(Bool())

    val bundleIn = Input(new Bundle2)
    val bundleOut = Output(new Bundle1)
  })
  
  //io.out := io.c // won't work due to different types

  // Okay, but Chisel will truncate the input width to 1 to match the output.
//   io.out := io.in

//   // Compiles; Chisel will connect the common subelements of the two Bundles (in this case, 'a').
//   io.bundleOut := io.bundleIn
}

println(getVerilog(new BadTypeModule))

---
# 类型 Generics<a name="类型-generics"></a>
Scala的泛型类型（也称为多态性）非常复杂，尤其是当与继承结合时。

本节只是让您初步了解；要了解更多，请查看[此教程](https://twitter.github.io/scala_school/类型-basics.html)。

类可以在其类型上具有多态性。一个很好的例子是序列，它需要知道其包含的类型。

In [ ]:
val seq1 = Seq("1", "2", "3") // Type is Seq[String]
val seq2 = Seq(1, 2, 3)       // Type is Seq[Int]
val seq3 = Seq(1, "2", true)  // Type is Seq[Any]

Sometimes, the Scala compiler needs help determining a polymorphic 类型, which requires the user to explicitly put the 类型:

In [ ]:
//val default = Seq() // Error!
val default = Seq[String]() // User must tell compiler that default is of type Seq[String]
Seq(1, "2", true).foldLeft(default){ (strings, next) =>
    next match {
        case s: String => strings ++ Seq(s)
        case _ => strings
    }
}

函数也可以在其输入或输出类型上具有多态性。以下示例定义了一个函数，用于计时运行代码块所需的时间。它基于代码块的返回类型进行参数化。*请注意，`=> T` 语法编码了一个没有参数列表的匿名函数，例如 `{ ... }` 与 `{ x => ... }`。*

In [ ]:
def time[T](block: => T): T = {
    val t0 = System.nanoTime()
    val result = block
    val t1 = System.nanoTime()
    val timeMillis = (t1 - t0) / 1000000.0
    println(s"Block took $timeMillis milliseconds!")
    result
}

// Adds 1 through a million
val int = time { (1 to 1000000).reduce(_ + _) }
println(s"Add 1 through a million is $int")

// Finds the largest number under a million that, in hex, contains "beef"
val string = time {
    (1 to 1000000).map(_.toHexString).filter(_.contains("beef")).last
}
println(s"The largest number under a million that has beef: $string")

## Chisel 类型 Hierarchy
To write 类型 generic code with Chisel, it is helpful to know a bit about the 类型 hierarchy of Chisel.

`chisel3.Data` is the base 类 for Chisel 硬件 types.
`UInt`, `SInt`, `Vec`, `束`, etc. are all instances of `Data`.
`Data` can be used in IOs and supports `:=`, wires, regs, etc.

Registers are a good 示例 of polymorphic code in Chisel.
Look at the 实现 of `RegEnable` (a 寄存器 with a `Bool` enable signal) [here](https://github.com/freechipsproject/chisel3/blob/v3.0.0/src/main/scala/chisel3/util/Reg.scala#L10).
apply 函数为 `[T <: Data]` 模板化，这意味着 `RegEnable` 适用于所有 Chisel 硬件类型。

Some operations are only defined on subtypes of `Bits`, 例如 `+`.
这就是为什么您可以添加 `UInt`s 或 `SInt`s，但不能添加 `束`s 或 `Vec`s。

<span style="color:blue">**示例: 类型 Generic ShiftRegister**<a name="类型-generic-shift-寄存器"></a></span><br>
在 Scala 中，对象和函数并不是我们可以视为参数的唯一事物。
我们也可以将类型视为参数。

我们通常需要提供一个类型约束。
在这种情况下，我们希望能够将对象放入束中，连接它们 (:=)，并用它们创建寄存器 (RegNext)。
这些操作不能对任意对象执行；例如，导线 := 3 是非法的，因为 3 是 Scala Int，而不是 Chisel UInt。
如果我们使用类型约束来声明类型 T 是 Data 的子类，那么我们可以对任何类型 T 的对象使用 :=，因为 := 是为所有 Data 定义的。

Here is an 实现 of a shift 寄存器 that take types as a 参数.
*gen* is an 参数 of 类型 T that tells what width to use, 例如 new ShiftRegister(UInt(4.W)) is a shift 寄存器 for 4-bit UInts.
*gen* also allows the Scala compiler to infer the 类型 T- you can write new ShiftRegister[UInt](UInt(4.W)) if you want to to be more specific, but the Scala compiler is smart enough to figure it out if you leave out the [UInt].

In [ ]:
class ShiftRegisterIO[T <: Data](gen: T, n: Int) extends Bundle {
    require (n >= 0, "Shift register must have non-negative shift")
    
    val in = Input(gen)
    val out = Output(Vec(n + 1, gen)) // + 1 because in is included in out
    override def cloneType: this.type = (new ShiftRegisterIO(gen, n)).asInstanceOf[this.type]
}

class ShiftRegister[T <: Data](gen: T, n: Int) extends Module {
    val io = IO(new ShiftRegisterIO(gen, n))
    
    io.out.foldLeft(io.in) { case (in, out) =>
        out := in
        RegNext(in)
    }
}

visualize(() => new ShiftRegister(SInt(6.W), 3))
test(new ShiftRegister(SInt(6.W), 3)) { c => 
    println(s"Testing ShiftRegister of type ${c.io.in} and depth ${c.io.out.length}")
    for (i <- 0 until 10) {
        c.io.in.poke(i.S) // magic literal creation
        println(s"$i: ${c.io.out.indices.map { index => c.io.out(index).peek().litValue} }")
        c.clock.step(1)
    }}

We generally recommend avoiding to use inheritance with 类型 generics.
It can be very tricky to do properly and can get frustrating quickly.

## 类型 Generics with Typeclasses

The 示例 above was limited to simple operations that could be performed on any 实例 of `Data` such as `:=` or `RegNext()`.
When generating DSP circuits, we would like to do mathematical operations like addition and multiplication.
The `dsptools` library provides tools for writing 类型 parameterized DSP generators.

Here is an 示例 of writing a multiply-accumulate 模块.
It can be used to generate a multiply-accumulate (MAC) for `FixedPoint`, `SInt`, or even `DspComplex[T]` (the complex number 类型 provided by `dsptools`).
The syntax of the 类型 bound is a little different because `dsptools` uses typeclasses.
They are beyond the scope of this notebook.
Read the `dsptools` readme and documentation for more information on using typeclasses.

`T <: Data : Ring` means that `T` is a subtype of `Data` and is also a `Ring` .
`Ring` is defined in `dsptools` as a number with `+` and `*` (among other operations).

_An alternative to `Ring` would be `Real`, but that would not allow us to make a MAC for `DspComplex()` because complex numbers are not `Real`._



In [ ]:
import chisel3.experimental._
import dsptools.numbers._

class Mac[T <: Data : Ring](genIn : T, genOut: T) extends Module {
    val io = IO(new Bundle {
        val a = Input(genIn)
        val b = Input(genIn)
        val c = Input(genIn)
        val out = Output(genOut)
    })
    io.out := io.a * io.b + io.c
}

println(getVerilog(new Mac(UInt(4.W), UInt(6.W)) ))
println(getVerilog(new Mac(SInt(4.W), SInt(6.W)) ))
println(getVerilog(new Mac(FixedPoint(4.W, 3.BP), FixedPoint(6.W, 4.BP))))


<span style="color:red">**练习: Mac as 对象**</span><br>

The Mac `模块` has a small number of inputs and just one 输出.
It might be convenient for other Chisel generators to write code like
```scala
val out = Mac(a, b, c)
```

Implement an `apply` 方法 in the `Mac` companion 对象 below that implements the `Mac` functionality.

In [ ]:
object Mac {
    def apply[T <: Data : Ring](a: T, b: T, c: T): T = {
        ??? // your code
    }
}

class MacTestModule extends Module {
    val io = IO(new Bundle {
        val uin = Input(UInt(4.W))
        val uout = Output(UInt())
        val sin = Input(SInt(4.W))
        val sout = Output(SInt())
        //val fin = Input(FixedPoint(16.W, 12.BP))
        //val fout = Output(FixedPoint())
    })
    // for each IO pair, do out = in * in + in
    io.uout := Mac(io.uin, io.uin, io.uin)
    io.sout := Mac(io.sin, io.sin, io.sin)
    //io.fout := Mac(io.fin, io.fin, io.fin)
}
println(getVerilog(new MacTestModule))

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong> (click to toggle displaying)</label>
<article>
<pre style="background-color:#f7f7f7">

        a * b + c

</pre></article></div></section></div>

<span style="color:red">**练习: Integrator**</span><br>
Implement an integrator as pictured below. $n_1$ is the width of `genReg` and $n_2$ is the width of `genIn`.

Don't forget that `Reg`, `RegInit`, `RegNext`, `RegEnable`, etc. are templated for types `T <: Data`.

<img src="images/integrator.svg" alt="Integrator" style="width: 250px;"/>

In [ ]:
class Integrator[T <: Data : Ring](genIn: T, genReg: T) extends Module {
    val io = IO(new Bundle {
        val in  = Input(genIn)
        val out = Output(genReg)
    })
    
    ??? // your code
}

test(new Integrator(SInt(4.W), SInt(8.W))) { c =>
    c.io.in.poke(3.S)
    c.io.out.expect(0.S)
    c.clock.step(1)
    c.io.in.poke(-4.S)
    c.io.out.expect(3.S)
    c.clock.step(1)
    c.io.in.poke(6.S)
    c.io.out.expect(-1.S)
    c.clock.step(1)
    c.io.out.expect(5.S)
}

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-2" />
<label for="check-2"><strong>解决方案</strong> (click to toggle displaying)</label>
<article>
<pre style="background-color:#f7f7f7">

类 Integrator\[T <: Data : Ring\](genIn: T, genReg: T) extends 模块 {
    val io = IO(new 束 {
        val in  = 输入(genIn.cloneType)
        val out = 输出(genReg.cloneType)
    })
    
    val reg = RegInit(genReg, Ring[T].zero) // init to zero
    reg := reg + io.in
    io.out := reg
}

</pre></article></div></section></div>

---
# Creating a Custom 类型<a name="creating-a-custom-类型"></a>

使 Chisel 强大的因素之一是其可扩展性。
您可以添加自己的类型，这些类型具有针对您的应用程序定制的操作和表示。
本节将介绍创建自定义类型的方法。

<span style="color:blue">**示例: DspComplex**</span><br>
`DspComplex` 是 **dsptools** 中定义的自定义数据类型[此处](https://github.com/ucb-bar/dsptools/blob/v1.0.0/src/main/scala/dsptools/numbers/chisel_concrete/DspComplex.scala#L59)。
需要理解的关键行是：
```scala
类 DspComplex[T <: Data:Ring](val real: T, val imag: T) extends 束 { ... }
```

`DspComplex` 是一个类型通用的容器。
这意味着复数的实部和虚部可以是任何类型，只要它们满足类型约束，即 `T <: Data : Ring`。

`T <: Data` 表示 `T` 是 `chisel3.Data` 的子类型，这是 Chisel 对象的基础类型。
这意味着 `DspComplex` 仅适用于 Chisel 类型的对象，而不适用于任意的 Scala 类型。

`T : Ring` 表示存在 `T` 的 Ring 类型类实现。
`Ring` 类型类定义了 `+` 和 `*` 运算符以及加法和乘法单位元（有关环的详细信息，请参阅[此维基百科文章](https://en.wikipedia.org/wiki/Ring_(mathematics))）。
**dsptools** 为常用的 Chisel 类型定义了类型类[此处](https://github.com/ucb-bar/dsptools/tree/v1.0.0/src/main/scala/dsptools/numbers/chisel_types)。

**dsptools** also defines a `Ring` typeclass for `DspComplex`, so we can reuse our MAC generator with complex numbers:

In [ ]:
println(getVerilog(new Mac(DspComplex(SInt(4.W), SInt(4.W)), DspComplex(SInt(6.W), SInt(6.W))) ))

<span style="color:red">**练习: Sign-magnitude Numbers**</span><br>
Suppose you wanted to use a sign-magnitude representation and want to reuse all of your DSP generators.
Typeclasses enable this kind of ad-hoc polymorphism.
以下 示例 gives the beggining of an 实现 of a SignMagnitude 类型 as well as an 实现 of a `Ring` typeclass that will allow the 类型 to be used with the Mac generator.

Fill in implementations for `+` and `*`.
You should pattern them after the 实现 for `unary_-()`.
The next block contains a 测试 that checks the correctness of a `Mac` that uses `SignMagnitude`.

In [ ]:
class SignMagnitude(val magnitudeWidth: Option[Int] = None) extends Bundle {
    val sign = Bool()
    val magnitude = magnitudeWidth match {
        case Some(w) => UInt(w.W)
        case None    => UInt()
    }
    def +(that: SignMagnitude): SignMagnitude = {
        // Implement this!
    }
    def -(that: SignMagnitude): SignMagnitude = {
        this.+(-that)
    }
    def unary_-(): SignMagnitude = {
        val result = Wire(new SignMagnitude())
        result.sign := !this.sign
        result.magnitude := this.magnitude
        result
    }
    def *(that: SignMagnitude): SignMagnitude = {
        // Implement this!
    }
}
trait SignMagnitudeRing extends Ring[SignMagnitude] {
    def plus(f: SignMagnitude, g: SignMagnitude): SignMagnitude = {
        f + g
    }
    def times(f: SignMagnitude, g: SignMagnitude): SignMagnitude = {
        f * g
    }
    def one: SignMagnitude = {
        val one = Wire(new SignMagnitude(Some(1)))
        one.sign := false.B
        one.magnitude := 1.U
        one
    }
    def zero: SignMagnitude = {
        val zero = Wire(new SignMagnitude(Some(0)))
        zero.sign := false.B
        zero.magnitude := 0.U
        zero
    }
    def negate(f: SignMagnitude): SignMagnitude = {
        -f
    }
    
    // Leave unimplemented for this example
    def minusContext(f: SignMagnitude, g: SignMagnitude): SignMagnitude = ???
    def negateContext(f: SignMagnitude): SignMagnitude = ???
    def plusContext(f: SignMagnitude,g: SignMagnitude): SignMagnitude = ???
    def timesContext(f: SignMagnitude,g: SignMagnitude): SignMagnitude = ???
}
implicit object SignMagnitudeRingImpl extends SignMagnitudeRing

In [ ]:
import chisel3.experimental.BundleLiterals._

test(new Mac(new SignMagnitude(Some(4)), new SignMagnitude(Some(5)))) { c =>
    c.io.a.poke(chiselTypeOf(c.io.a).Lit(_.sign -> false.B, _.magnitude -> 3.U))
    c.io.b.poke(chiselTypeOf(c.io.b).Lit(_.sign -> false.B, _.magnitude -> 3.U))
    c.io.c.poke(chiselTypeOf(c.io.c).Lit(_.sign -> false.B, _.magnitude -> 2.U))
    c.io.out.expect(chiselTypeOf(c.io.out).Lit(_.sign -> false.B, _.magnitude -> 11.U))

    c.io.c.sign.poke(true.B)
    c.io.out.expect(chiselTypeOf(c.io.out).Lit(_.sign -> false.B, _.magnitude -> 7.U))

    c.io.b.sign.poke(true.B)
    c.io.out.expect(chiselTypeOf(c.io.out).Lit(_.sign -> true.B, _.magnitude -> 11.U))
}
println("SUCCESS!!") // Scala Code: if we get here, our tests passed!

Look at the verilog to see if the 输出 looks reasonable:

In [ ]:
println(getVerilog(new Mac(new SignMagnitude(Some(4)), new SignMagnitude(Some(5)))))

`SignMagnitude` even works with `DspComplex`!

In [ ]:
println(getVerilog(new Mac(DspComplex(new SignMagnitude(Some(4)), new SignMagnitude(Some(4))), DspComplex(new SignMagnitude(Some(5)), new SignMagnitude(Some(5))))))

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-3" />
<label for="check-3"><strong>解决方案</strong> (click to toggle displaying)</label>
<article>
<pre style="background-color:#f7f7f7">
    // implementations for 类 SignMagnitude

    def +(that: SignMagnitude): SignMagnitude = {
      val result = 导线(new SignMagnitude())
      val signsTheSame = this.sign === that.sign
      when (signsTheSame) {
        result.sign      := this.sign
        result.magnitude := this.magnitude + that.magnitude
      } .otherwise {
        when (this.magnitude > that.magnitude) {
          result.sign      := this.sign
          result.magnitude := this.magnitude - that.magnitude
        } .otherwise {
          result.sign      := that.sign
          result.magnitude := that.magnitude - this.magnitude
        }   
      }   
      result
    }
    def *(that: SignMagnitude): SignMagnitude = {
        val result = 导线(new SignMagnitude())
        result.sign := this.sign ^ that.sign
        result.magnitude := this.magnitude * that.magnitude
        result
    }


</pre></article></div></section></div>